In [1]:
import pandas as pd
import time
import joblib
import matplotlib.pyplot as plt
import xgboost as xgb
from xgboost import XGBClassifier, plot_importance
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, RocCurveDisplay


In [2]:

# -----------------------------
# LOAD DATA
# -----------------------------
df = pd.read_parquet("./data/training_data.parquet")

print("Loaded data shape:", df.shape)
print("Columns:", list(df.columns))


Loaded data shape: (423626, 15)
Columns: ['AF', 'PR', 'T2', 'D2', 'WS', 'FU_LL', 'FU_LW', 'FU_DF', 'FU_DW', 'DF', 'DW', 'LF', 'UR', 'PO', 'RD']


In [3]:

# -----------------------------
# DEFINE FEATURES & TARGET
# -----------------------------
target_col = "AF"
feature_cols = [c for c in df.columns if c not in ["AF"]]

X = df[feature_cols]
y = df[target_col]


In [4]:

# -----------------------------
# TRAIN / TEST SPLIT
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [5]:

# -----------------------------
# TRAIN CLASSIFIER
# -----------------------------
print("Training XGBoost Binary Classifier...")
start_time = time.time()

model = XGBClassifier(
    objective="binary:logistic",
    tree_method="hist",
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
#    subsample=0.8,
#    colsample_bytree=0.8,
    eval_metric="logloss",
#    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

end_time = time.time()
print(f"✅ Training completed in {end_time - start_time:.2f} seconds")


Training XGBoost Binary Classifier...
✅ Training completed in 1.24 seconds


In [6]:

# -----------------------------
# EVALUATE MODEL
# -----------------------------
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("\nClassification report:")
print(classification_report(y_test, y_pred, digits=3))

roc_auc = roc_auc_score(y_test, y_proba)
print(f"ROC-AUC: {roc_auc:.3f}")

# Confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))



Classification report:
              precision    recall  f1-score   support

         0.0      0.998     1.000     0.999     84542
         1.0      0.000     0.000     0.000       184

    accuracy                          0.998     84726
   macro avg      0.499     0.500     0.499     84726
weighted avg      0.996     0.998     0.997     84726

ROC-AUC: 0.954

Confusion Matrix:
[[84538     4]
 [  184     0]]


In [7]:

# -----------------------------
# PLOT ROC CURVE & FEATURE IMPORTANCE
# -----------------------------
plt.figure(figsize=(6, 5))
RocCurveDisplay.from_estimator(model, X_test, y_test)
plt.title("ROC Curve")
plt.savefig("./images/POF_ROC.png", dpi=300)
plt.close()

plt.figure(figsize=(10, 6))
plot_importance(model, max_num_features=20, importance_type="gain")
plt.title("Feature Importance (Gain)")
plt.tight_layout()
plt.savefig("./images/POF_importance.png", dpi=300)
plt.close()


<Figure size 600x500 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

In [8]:

# -----------------------------
# SAVE MODEL
# -----------------------------
out_model = "./data/POF_model.joblib"
joblib.dump(model, out_model, compress=3)
print(f"Model saved → {out_model}")

Model saved → ./data/POF_model.joblib
